In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the configuration
with open('final_features_config.json', 'r') as f:
    config = json.load(f)

# 2. Load data and filter
df = pd.read_csv('../data/application_train.csv')
# Keep only selected features + Target
df_final = df[config['final_features'] + [config['target']]].copy()

print(f"Data loaded with {df_final.shape[1]} columns.")

Data loaded with 40 columns.


In [5]:
# Cell 2: Investigate Missing Data and Types
import pandas as pd

# 1. Calculate missing rates
missing_counts = df_final.isnull().sum()
missing_percent = (missing_counts / len(df_final)) * 100
missing_df = pd.DataFrame({'Missing Rate (%)': missing_percent}).loc[missing_counts > 0]
missing_df = missing_df.sort_values(by='Missing Rate (%)', ascending=False)

print("--- Data Investigation for Imputation ---")

for col in missing_df.index:
    # Use pd.api.types for more reliable type checking
    if pd.api.types.is_numeric_dtype(df_final[col]):
        print(f"[Numeric] {col}")
        print(f"  -> Missing: {missing_df.loc[col, 'Missing Rate (%)']:.2f}%")
        print(f"  -> Median: {df_final[col].median()}")
    else:
        print(f"[Categorical] {col}")
        print(f"  -> Missing: {missing_df.loc[col, 'Missing Rate (%)']:.2f}%")
        # Show actual unique values so we know what to expect in the React form
        unique_vals = df_final[col].dropna().unique()
        print(f"  -> Options: {list(unique_vals)}")
    print("-" * 30)

--- Data Investigation for Imputation ---
[Categorical] FONDKAPREMONT_MODE
  -> Missing: 68.39%
  -> Options: ['reg oper account', 'org spec account', 'reg oper spec account', 'not specified']
------------------------------
[Numeric] OWN_CAR_AGE
  -> Missing: 65.99%
  -> Median: 9.0
------------------------------
[Categorical] WALLSMATERIAL_MODE
  -> Missing: 50.84%
  -> Options: ['Stone, brick', 'Block', 'Panel', 'Mixed', 'Wooden', 'Others', 'Monolithic']
------------------------------
[Categorical] HOUSETYPE_MODE
  -> Missing: 50.18%
  -> Options: ['block of flats', 'terraced house', 'specific housing']
------------------------------
[Categorical] EMERGENCYSTATE_MODE
  -> Missing: 47.40%
  -> Options: ['No', 'Yes']
------------------------------
[Categorical] OCCUPATION_TYPE
  -> Missing: 31.35%
  -> Options: ['Laborers', 'Core staff', 'Accountants', 'Managers', 'Drivers', 'Sales staff', 'Cleaning staff', 'Cooking staff', 'Private service staff', 'Medicine staff', 'Security staff', '

In [8]:
# Cell 3: Execute Imputation Strategy

# Create a copy to track changes
df_clean = df_final.copy()

# 1. Categorical Imputation
cat_cols_missing = df_clean.select_dtypes(include=['object', 'string']).columns

for col in cat_cols_missing:
    if df_clean[col].isnull().sum() > 0:
        # For high missingness categorical, 'Unknown' is a better signal than the mode
        df_clean[col] = df_clean[col].fillna("Unknown")

# 2. Numeric Imputation
# Special Case: OWN_CAR_AGE (Missing usually means no car)
if 'OWN_CAR_AGE' in df_clean.columns:
    df_clean['OWN_CAR_AGE'] = df_clean['OWN_CAR_AGE'].fillna(0)

# General Case: Fill other numerics with Median
num_cols_missing = df_clean.select_dtypes(include=['number']).columns
for col in num_cols_missing:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

print("Imputation Complete!")
print(f"Total Missing Values remaining: {df_clean.isnull().sum().sum()}")

Imputation Complete!
Total Missing Values remaining: 0


In [9]:
# Cell 4: Save the Imputed Dataset to Disk
df_clean.to_csv('application_train_imputed.csv', index=False)

print("Success! The imputed dataset is now physically stored as 'application_train_imputed.csv'")

Success! The imputed dataset is now physically stored as 'application_train_imputed.csv'
